In [ ]:
import os
import sys

import torch
import torch.nn.functional as F
import cv2
import numpy as np
from torchvision import transforms

sys.path.append(os.path.abspath('./Matcher'))
from dinov2.models import vision_transformer as vits
import dinov2.utils.utils as dinov2_utils

print("MATCHER 로컬 모듈 로드 완료. 모델 생성 중...")

dinov2_kwargs = dict(
    img_size=518,
    patch_size=14,
    init_values=1e-5,
    ffn_layer='mlp',
    block_chunks=0,
    qkv_bias=True,
    proj_bias=True,
    ffn_bias=True,
)

model = vits.__dict__['vit_large'](**dinov2_kwargs)

weights_path = 'model/dinov2_vitl14_pretrain.pth'
dinov2_utils.load_pretrained_weights(model, weights_path, 'teacher')

model = model.cuda().eval()
print("GPU 세팅 끝")

preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((518,518)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

def create_anchor_feature(image_path, mask_path, model, save_path):
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"이미지 없음: {image_path}")
        
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    image_tensor = preprocess(image).unsqueeze(0).cuda()

    with torch.no_grad(): # 로컬 모듈은 구형 환경에서도 여기서 터지지 않음
        # MATCHER는 x_norm_patchtokens 대신 x_prenorm을 주로 사용함
        feature = model.forward_features(image_tensor)["x_prenorm"][:, 1:]

        c = feature.shape[-1]
        h = w = int(np.sqrt(feature.shape[1]))
        features = feature.permute(0, 2, 1).reshape(1, c, h, w)

    mask_tensor = torch.tensor(mask).unsqueeze(0).unsqueeze(0).float().cuda()
    mask_resized = F.interpolate(mask_tensor, size=(h,w), mode='nearest')
    mask_resized = mask_resized / 255.0
    
    masked_feature = features * mask_resized
    tumor_vector = masked_feature.sum(dim=(2,3)) / (mask_resized.sum(dim=(2,3)) + 1e-6)

    print(f"Extracted feature shape: {tumor_vector.shape}")
    
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    torch.save(tumor_vector.cpu(), save_path)
    print(f"Saved anchor feature to {save_path}")


create_anchor_feature(
    image_path='../BUSI_test_set/image/malignant (206).png', 
    mask_path='../BUSI_test_set/groundtruth/malignant (206)_mask.png', 
    model=model, 
    save_path='save_image/anchor_malignant.pt'
)